# 10 - Background Comparison
Train the same EVA configuration on `../data` and `../data_background`, enforce a shared validation split, and run a 2x2 cross-evaluation so we can isolate whether the hidden background RGB values matter.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.wandb_utils as wandb_utils
from src.models import ArcFaceModel, load_arcface_model_from_checkpoint
from src.training import build_optimizer, fit, compute_map_from_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

device = get_device()
device


device(type='cuda')

## Config


In [ ]:
BASE_RUN_NAME = "eva_unfrozen_rs_04_hlr1e-04_blr1e-05_wd1e-05_do0.2_aug1_bs16"

config = {
    "dataset_sources": [
        {"source_name": "jaguars2", "path": Path("../data_background")},
        {"source_name": "data", "path": Path("../data")},
    ],
    "checkpoint_dir": Path("../checkpoints/e15_dataset_source_comparison"),
    "submissions_dir": Path("../submissions/e15_dataset_source_comparison"),
    "results_path": Path("../checkpoints/e15_dataset_source_comparison/dataset_source_results.csv"),
    "cross_eval_results_path": Path("../checkpoints/e15_dataset_source_comparison/cross_eval_results.csv"),
    "generate_submissions": True,
    "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    "input_size": 448,
    "freeze_backbone": False,
    "train_last_n_layers": 0,
    "embedding_dim": 384,
    "hidden_dim": 768,
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.2,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "head_learning_rate": 1e-4,
    "backbone_learning_rate": 1e-5,
    "weight_decay": 1e-5,
    "num_epochs": 25,
    "patience": 8,
    "scheduler_patience": 2,
    "val_split": 0.2,
    "num_workers": 2,
    "augment_train": True,
    "seed": 42,
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
    "rerank": {
        "enabled": True,
        "k1": 20,
        "k2": 6,
        "lambda_value": 0.3,
    },
}

config["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
config["submissions_dir"].mkdir(parents=True, exist_ok=True)
config


## Helpers


In [3]:
def resolve_dataset_dir(source_path: Path) -> Path:
    source_path = source_path.expanduser().resolve()
    if not (source_path / "train.csv").exists() or not (source_path / "test.csv").exists():
        raise FileNotFoundError(f"Expected train.csv and test.csv in {source_path}")
    return source_path


def load_source_bundle(source_config):
    data_dir = resolve_dataset_dir(source_config["path"])
    train_df = data.load_train_df(data_dir)
    test_pairs_df = data.load_test_pairs_df(data_dir)
    return {
        "source_name": source_config["source_name"],
        "data_dir": data_dir,
        "train_df": train_df,
        "test_pairs_df": test_pairs_df,
    }


def compare_sources(reference_bundle, other_bundle):
    reference_train = reference_bundle["train_df"][["filename", "ground_truth"]].copy()
    other_train = other_bundle["train_df"][["filename", "ground_truth"]].copy()
    reference_test = reference_bundle["test_pairs_df"][["query_image", "gallery_image"]].copy()
    other_test = other_bundle["test_pairs_df"][["query_image", "gallery_image"]].copy()

    reference_labels = reference_train.set_index("filename")["ground_truth"].sort_index()
    other_labels = other_train.set_index("filename")["ground_truth"].sort_index()
    reference_test_images = set(reference_test["query_image"]) | set(reference_test["gallery_image"])
    other_test_images = set(other_test["query_image"]) | set(other_test["gallery_image"])

    return {
        "reference_source": reference_bundle["source_name"],
        "other_source": other_bundle["source_name"],
        "train_rows_match": len(reference_train) == len(other_train),
        "test_rows_match": len(reference_test) == len(other_test),
        "train_csv_identical": reference_train.equals(other_train),
        "test_csv_identical": reference_test.equals(other_test),
        "same_train_filename_set": set(reference_train["filename"]) == set(other_train["filename"]),
        "same_labels_by_filename": reference_labels.equals(other_labels),
        "same_test_image_set": reference_test_images == other_test_images,
    }


def build_shared_split(reference_train_df, val_split, seed):
    canonical_df = reference_train_df[["filename", "ground_truth"]].copy()
    canonical_df, label_encoder = data.encode_labels(canonical_df)
    train_split_df, val_split_df = data.train_val_split(
        canonical_df,
        val_split=val_split,
        seed=seed,
        stratify_col="ground_truth",
    )
    return {
        "canonical_df": canonical_df.reset_index(drop=True),
        "train_df": train_split_df.reset_index(drop=True),
        "val_df": val_split_df.reset_index(drop=True),
        "label_encoder": label_encoder,
    }


def create_loaders_for_source(run_config, shared_split, source_bundle):
    train_loader, val_loader = data.create_dataloaders(
        shared_split["train_df"],
        shared_split["val_df"],
        img_dir=source_bundle["data_dir"] / "train" / "train",
        input_size=run_config["input_size"],
        batch_size=run_config["batch_size"],
        num_workers=run_config["num_workers"],
        mean=run_config["mean"],
        std=run_config["std"],
        weighted_sampling=False,
        augment=run_config["augment_train"],
    )
    return train_loader, val_loader


def build_run_config(base_config, source_bundle):
    run_config = {k: v for k, v in base_config.items() if k not in {"dataset_sources"}}
    run_config["source_name"] = source_bundle["source_name"]
    run_config["data_dir"] = source_bundle["data_dir"]
    run_config["name"] = f"{BASE_RUN_NAME}__{source_bundle['source_name']}"
    run_config["learning_rate"] = run_config["head_learning_rate"]
    return run_config


def build_model_for_run(run_config, num_classes):
    return ArcFaceModel(
        backbone_name=run_config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=run_config["embedding_dim"],
        hidden_dim=run_config["hidden_dim"],
        margin=run_config["arcface_margin"],
        scale=run_config["arcface_scale"],
        dropout=run_config["dropout"],
        pretrained=True,
        freeze_backbone=run_config["freeze_backbone"],
        train_last_n_layers=run_config["train_last_n_layers"],
    ).to(device)


def log_comparison_tables_to_wandb(base_config, source_compatibility_df, shared_split, results_df, cross_eval_df):
    summary_config = {k: v for k, v in base_config.items() if k != "dataset_sources"}
    summary_config["summary_type"] = "background_comparison"
    wandb_run = wandb_utils.init_wandb(
        summary_config,
        run_name="background_comparison_summary",
        param_count=0,
        param_count_backbone=0,
    )
    wandb_run.log({
        "source_compatibility": wandb_run.Table(dataframe=source_compatibility_df),
        "dataset_source_results": wandb_run.Table(dataframe=results_df),
        "cross_eval_results": wandb_run.Table(dataframe=cross_eval_df),
    })
    wandb_run.run.summary["shared_train_rows"] = len(shared_split["train_df"])
    wandb_run.run.summary["shared_val_rows"] = len(shared_split["val_df"])
    if not results_df.empty:
        best_row = results_df.sort_values(["best_map_rerank", "best_map"], ascending=False).iloc[0]
        wandb_run.run.summary["best_source_name"] = str(best_row["source_name"])
        wandb_run.run.summary["best_source_val_map"] = float(best_row["best_map"])
        wandb_run.run.summary["best_source_val_map_rerank"] = float(best_row["best_map_rerank"])
    if not cross_eval_df.empty:
        best_cross_row = cross_eval_df.sort_values("val_map_rerank", ascending=False).iloc[0]
        wandb_run.run.summary["best_cross_train_source"] = str(best_cross_row["train_source"])
        wandb_run.run.summary["best_cross_eval_source"] = str(best_cross_row["eval_source"])
        wandb_run.run.summary["best_cross_val_map"] = float(best_cross_row["val_map"])
        wandb_run.run.summary["best_cross_val_map_rerank"] = float(best_cross_row["val_map_rerank"])
    wandb_run.finish()


def build_test_loader(model, checkpoint_config, batch_size, num_workers, seed):
    data_dir = Path(checkpoint_config["data_dir"])
    pairs_df = data.load_test_pairs_df(data_dir)
    unique_images = sorted(set(pairs_df["query_image"]) | set(pairs_df["gallery_image"]))
    test_df = pd.DataFrame({"filename": unique_images})
    transform = data.build_transforms(model.backbone, int(checkpoint_config["input_size"]))
    test_loader = data.create_eval_loader(
        test_df,
        data_dir / "test" / "test",
        transform,
        batch_size=batch_size,
        num_workers=num_workers,
        is_test=True,
        seed=seed,
    )
    return pairs_df, test_df, test_loader


def generate_submissions_for_checkpoint(checkpoint_path, output_dir, batch_size=None, num_workers=None):
    model, checkpoint, checkpoint_config = load_arcface_model_from_checkpoint(checkpoint_path, device)
    effective_batch_size = batch_size or int(checkpoint_config.get("batch_size", 32))
    effective_num_workers = int(checkpoint_config.get("num_workers", 2) if num_workers is None else num_workers)
    seed = int(checkpoint_config.get("seed", 42))
    rerank_config = checkpoint_config.get("rerank", {"enabled": False, "k1": 20, "k2": 6, "lambda_value": 0.3})

    pairs_df, test_df, test_loader = build_test_loader(
        model,
        checkpoint_config,
        batch_size=effective_batch_size,
        num_workers=effective_num_workers,
        seed=seed,
    )

    plain_output = output_dir / f"{checkpoint_path.stem}_submission.csv"
    rerank_output = output_dir / f"{checkpoint_path.stem}_submission_rerank.csv"

    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=plain_output,
        use_rerank=False,
    )
    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=rerank_output,
        use_rerank=rerank_config.get("enabled", False),
        k1=rerank_config.get("k1", 20),
        k2=rerank_config.get("k2", 6),
        lambda_value=rerank_config.get("lambda_value", 0.3),
    )

    return {
        "submission_checkpoint_epoch": checkpoint.get("epoch"),
        "submission_data_dir": checkpoint_config.get("data_dir"),
        "plain_submission_path": plain_output,
        "rerank_submission_path": rerank_output,
        "num_test_pairs": len(pairs_df),
        "num_unique_test_images": len(test_df),
    }


def run_experiment(run_config, source_bundle, shared_split, run_idx, total_runs):
    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {run_config['name']}")
    print(f"Data dir: {run_config['data_dir']}")

    set_seed(run_config["seed"])
    train_loader, val_loader = create_loaders_for_source(run_config, shared_split, source_bundle)
    model = build_model_for_run(run_config, len(shared_split["label_encoder"].classes_))

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone_params = sum(p.numel() for p in model.backbone.parameters())

    wandb_run = wandb_utils.init_wandb(
        run_config,
        run_name=run_config["name"],
        param_count=trainable_params,
        param_count_backbone=backbone_params,
    )
    wandb_run.run.summary["total_param_count"] = total_params
    wandb_run.run.summary["trainable_param_count"] = trainable_params
    wandb_run.run.summary["source_run_index"] = run_idx
    wandb_run.run.summary["source_run_total"] = total_runs
    wandb_run.run.summary["shared_train_rows"] = len(shared_split["train_df"])
    wandb_run.run.summary["shared_val_rows"] = len(shared_split["val_df"])

    checkpoint_path = run_config["checkpoint_dir"] / f"{run_config['name']}.pth"
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, run_config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=run_config["scheduler_patience"],
    )

    try:
        results = fit(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=run_config,
            device=device,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            label_encoder=shared_split["label_encoder"],
            wandb_run=wandb_run,
            checkpoint_name=checkpoint_path.name,
        )

        submission_artifacts = None
        if config["generate_submissions"]:
            submission_artifacts = generate_submissions_for_checkpoint(
                checkpoint_path,
                output_dir=run_config["submissions_dir"],
                batch_size=run_config["batch_size"],
                num_workers=run_config["num_workers"],
            )

        run_result = {
            "source_name": source_bundle["source_name"],
            "data_dir": str(source_bundle["data_dir"]),
            "train_rows": len(source_bundle["train_df"]),
            "num_classes": len(shared_split["label_encoder"].classes_),
            "train_split_rows": len(shared_split["train_df"]),
            "val_split_rows": len(shared_split["val_df"]),
            "checkpoint_path": str(checkpoint_path),
            "best_val_loss": results["best_val_loss"],
            "best_map": results["best_map"],
            "best_map_rerank": results["best_map_rerank"],
            "best_epoch": results["best_epoch"],
            "epochs_ran": results["epochs_ran"],
            "checkpoint_metric": results["best_checkpoint_metric"],
            "checkpoint_metric_name": results["best_checkpoint_metric_name"],
            "submission_checkpoint_epoch": None if submission_artifacts is None else submission_artifacts["submission_checkpoint_epoch"],
            "submission_data_dir": None if submission_artifacts is None else submission_artifacts["submission_data_dir"],
            "num_test_pairs": None if submission_artifacts is None else submission_artifacts["num_test_pairs"],
            "num_unique_test_images": None if submission_artifacts is None else submission_artifacts["num_unique_test_images"],
            "plain_submission_path": None if submission_artifacts is None else str(submission_artifacts["plain_submission_path"]),
            "rerank_submission_path": None if submission_artifacts is None else str(submission_artifacts["rerank_submission_path"]),
        }

        wandb_run.run.summary["best_val_mAP"] = run_result["best_map"]
        wandb_run.run.summary["best_val_mAP_rerank"] = run_result["best_map_rerank"]
        wandb_run.run.summary["best_val_loss"] = run_result["best_val_loss"]
        wandb_run.run.summary["best_epoch"] = run_result["best_epoch"]
        wandb_run.run.summary["total_epochs"] = run_result["epochs_ran"]
        wandb_run.log({"run_result": wandb_run.Table(dataframe=pd.DataFrame([run_result]))})
        wandb_utils.log_checkpoint_artifact(
            wandb_run,
            checkpoint_path,
            artifact_name=run_config["name"],
            description="background comparison checkpoint",
        )
        if submission_artifacts is not None:
            wandb_utils.log_submission_artifact(
                wandb_run,
                Path(run_result["plain_submission_path"]),
                artifact_name=f"{run_config['name']}_plain_submission",
                description="background comparison plain submission",
            )
            wandb_utils.log_submission_artifact(
                wandb_run,
                Path(run_result["rerank_submission_path"]),
                artifact_name=f"{run_config['name']}_rerank_submission",
                description="background comparison reranked submission",
            )
        return run_result
    finally:
        wandb_run.finish()


def evaluate_checkpoint_on_source(checkpoint_path, eval_source_bundle, shared_split, batch_size, num_workers):
    model, checkpoint, checkpoint_config = load_arcface_model_from_checkpoint(checkpoint_path, device)
    rerank_config = checkpoint_config.get("rerank", {"enabled": False, "k1": 20, "k2": 6, "lambda_value": 0.3})
    seed = int(checkpoint_config.get("seed", config["seed"]))
    transform = data.build_transforms(model.backbone, int(checkpoint_config["input_size"]))
    val_loader = data.create_eval_loader(
        shared_split["val_df"],
        eval_source_bundle["data_dir"] / "train" / "train",
        transform,
        batch_size=batch_size,
        num_workers=num_workers,
        is_test=False,
        seed=seed,
    )
    val_embeddings, val_labels = inference.extract_embeddings_with_labels(model, val_loader, device)
    val_map = compute_map_from_embeddings(val_embeddings, val_labels)
    val_map_rerank = compute_map_from_embeddings(
        val_embeddings,
        val_labels,
        use_rerank=rerank_config.get("enabled", False),
        k1=rerank_config.get("k1", 20),
        k2=rerank_config.get("k2", 6),
        lambda_value=rerank_config.get("lambda_value", 0.3),
    )
    return {
        "checkpoint_epoch": checkpoint.get("epoch"),
        "checkpoint_data_dir": checkpoint_config.get("data_dir"),
        "val_map": val_map,
        "val_map_rerank": val_map_rerank,
    }


## Resolve And Verify Sources


In [4]:
source_bundles = [load_source_bundle(source_config) for source_config in config["dataset_sources"]]

source_overview_df = pd.DataFrame(
    {
        "source_name": bundle["source_name"],
        "data_dir": str(bundle["data_dir"]),
        "train_rows": len(bundle["train_df"]),
        "identities": bundle["train_df"]["ground_truth"].nunique(),
        "test_pairs": len(bundle["test_pairs_df"]),
        "unique_test_images": len(set(bundle["test_pairs_df"]["query_image"]) | set(bundle["test_pairs_df"]["gallery_image"])),
    }
    for bundle in source_bundles
)

reference_bundle = source_bundles[0]
source_compatibility_df = pd.DataFrame(
    compare_sources(reference_bundle, bundle)
    for bundle in source_bundles[1:]
)

source_overview_df


,source_name,data_dir,train_rows,identities,test_pairs,unique_test_images
0,jaguars2,/sc/home/maximilian.speer/vs/jaguars/data_back...,1895,31,137270,371
1,data,/sc/home/maximilian.speer/vs/jaguars/data,1895,31,137270,371


In [5]:
source_compatibility_df


,reference_source,other_source,train_rows_match,test_rows_match,train_csv_identical,test_csv_identical,same_train_filename_set,same_labels_by_filename,same_test_image_set
0,jaguars2,data,True,True,True,True,True,True,True


## Shared Split


In [6]:
shared_split = build_shared_split(
    reference_bundle["train_df"],
    val_split=config["val_split"],
    seed=config["seed"],
)

pd.DataFrame(
    [
        {
            "train_split_rows": len(shared_split["train_df"]),
            "val_split_rows": len(shared_split["val_df"]),
            "num_classes": len(shared_split["label_encoder"].classes_),
            "seed": config["seed"],
            "val_split": config["val_split"],
        }
    ]
)


,train_split_rows,val_split_rows,num_classes,seed,val_split
0,1516,379,31,42,0.2


## Train Or Reuse Checkpoints


In [7]:
experiment_results = []

for run_idx, source_bundle in enumerate(source_bundles, start=1):
    run_config = build_run_config(config, source_bundle)
    result = run_experiment(run_config, source_bundle, shared_split, run_idx, len(source_bundles))
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results).sort_values(["best_map_rerank", "best_map"], ascending=False).reset_index(drop=True)
results_df.to_csv(config["results_path"], index=False)
results_df


Run 1/2: eva_unfrozen_rs_04_hlr1e-04_blr1e-05_wd1e-05_do0.2_aug1_bs16__jaguars2
Data dir: /sc/home/maximilian.speer/vs/jaguars/data_background
Reusing checkpoint for jaguars2: ../checkpoints/e15_dataset_source_comparison/eva_unfrozen_rs_04_hlr1e-04_blr1e-05_wd1e-05_do0.2_aug1_bs16__jaguars2.pth


Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

...

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Building submission scores:   0%|          | 0/137270 [00:00<?, ?it/s]

,source_name,data_dir,train_rows,num_classes,train_split_rows,val_split_rows,checkpoint_reused,checkpoint_path,best_val_loss,best_map,...,best_epoch,epochs_ran,checkpoint_metric,checkpoint_metric_name,submission_checkpoint_epoch,submission_data_dir,num_test_pairs,num_unique_test_images,plain_submission_path,rerank_submission_path
0,data,/sc/home/maximilian.speer/vs/jaguars/data,1895,31,1516,379,True,../checkpoints/e15_dataset_source_comparison/e...,2.259301,0.892090,...,7,7,0.916803,val_map_rerank,7,/sc/home/maximilian.speer/vs/jaguars/data,137270,371,../submissions/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...
1,jaguars2,/sc/home/maximilian.speer/vs/jaguars/data_back...,1895,31,1516,379,True,../checkpoints/e15_dataset_source_comparison/e...,2.493482,0.898283,...,18,18,0.906509,val_map_rerank,18,/sc/home/maximilian.speer/vs/jaguars/data_back...,137270,371,../submissions/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...


## Cross-Evaluate On The Shared Validation Split


In [8]:
cross_eval_rows = []

for train_result in experiment_results:
    checkpoint_path = Path(train_result["checkpoint_path"])
    for eval_source_bundle in source_bundles:
        metrics = evaluate_checkpoint_on_source(
            checkpoint_path,
            eval_source_bundle,
            shared_split,
            batch_size=config["batch_size"],
            num_workers=config["num_workers"],
        )
        cross_eval_rows.append(
            {
                "train_source": train_result["source_name"],
                "eval_source": eval_source_bundle["source_name"],
                "checkpoint_path": str(checkpoint_path),
                "checkpoint_epoch": metrics["checkpoint_epoch"],
                "checkpoint_data_dir": metrics["checkpoint_data_dir"],
                "eval_data_dir": str(eval_source_bundle["data_dir"]),
                "shared_val_rows": len(shared_split["val_df"]),
                "val_map": metrics["val_map"],
                "val_map_rerank": metrics["val_map_rerank"],
            }
        )

cross_eval_df = pd.DataFrame(cross_eval_rows).sort_values(["train_source", "eval_source"]).reset_index(drop=True)
cross_eval_df.to_csv(config["cross_eval_results_path"], index=False)
log_comparison_tables_to_wandb(config, source_compatibility_df, shared_split, results_df, cross_eval_df)
cross_eval_df


Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

,train_source,eval_source,checkpoint_path,checkpoint_epoch,checkpoint_data_dir,eval_data_dir,shared_val_rows,val_map,val_map_rerank
0,data,data,../checkpoints/e15_dataset_source_comparison/e...,7,/sc/home/maximilian.speer/vs/jaguars/data,/sc/home/maximilian.speer/vs/jaguars/data,379,0.892066,0.912223
1,data,jaguars2,../checkpoints/e15_dataset_source_comparison/e...,7,/sc/home/maximilian.speer/vs/jaguars/data,/sc/home/maximilian.speer/vs/jaguars/data_back...,379,0.602109,0.608503
2,jaguars2,data,../checkpoints/e15_dataset_source_comparison/e...,18,/sc/home/maximilian.speer/vs/jaguars/data_back...,/sc/home/maximilian.speer/vs/jaguars/data,379,0.818686,0.830932
3,jaguars2,jaguars2,../checkpoints/e15_dataset_source_comparison/e...,18,/sc/home/maximilian.speer/vs/jaguars/data_back...,/sc/home/maximilian.speer/vs/jaguars/data_back...,379,0.898359,0.906740


## Summary


In [9]:
results_df[[
    "source_name",
    "checkpoint_reused",
    "data_dir",
    "best_map",
    "best_map_rerank",
    "best_val_loss",
    "best_epoch",
    "checkpoint_path",
    "plain_submission_path",
    "rerank_submission_path",
]]


,source_name,checkpoint_reused,data_dir,best_map,best_map_rerank,best_val_loss,best_epoch,checkpoint_path,plain_submission_path,rerank_submission_path
0,data,True,/sc/home/maximilian.speer/vs/jaguars/data,0.892090,0.916803,2.259301,7,../checkpoints/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...
1,jaguars2,True,/sc/home/maximilian.speer/vs/jaguars/data_back...,0.898283,0.906509,2.493482,18,../checkpoints/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...,../submissions/e15_dataset_source_comparison/e...


In [10]:
cross_eval_df.pivot(index="train_source", columns="eval_source", values="val_map_rerank")


eval_source,data,jaguars2
train_source,,
data,0.912223,0.608503
jaguars2,0.830932,0.906740


View the runs on W&B: [W&B Run Group](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/groups/Experiment-10-Background/)

![](../images/e10_wandb_dashboard.png)